# PHASE 4: DATA CLEANING AND MISSING VALUE IMPUTATION

## Objectives
- Load the 250-row merged dataset from Phase 3.
- Normalize the `Location` column by mapping free-text entries to Vietnam's 63 official provinces.
- Parse the raw salary field (`Salary_Raw`) into two numeric columns, `Salary_Min_VND` and `Salary_Max_VND`, with all values converted to monthly VND.
- Impute missing values in salary and experience fields using a two-tier Groupby Median strategy under a MAR assumption.

In [14]:
import pandas as pd
import numpy as np
import re
import os

# Load the 250-row merged dataset from Phase 3
DATA_PATH = "../data/03_interim/250_merged_raw.csv" 
df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns.")
print("Current columns:", df.columns.tolist())

Dataset loaded successfully: 250 rows, 10 columns.
Current columns: ['Job_Title', 'Job Domain', 'Company', 'Location', 'Min years of exp', 'Language Requirement', 'Skills', 'Salary_Raw', 'Link', 'JD & Requirements']


In [15]:
# Normalize Location column to standard province names
# Function to remove Vietnamese diacritics
def remove_diacritics(text):
    if pd.isna(text): return text
    text = str(text)
    s1 = u'ÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝàáâãèéêìíòóôõùúýĂăĐđĨĩŨũƠơƯưẠạẢảẤấẦầẨẩẪẫẬậẮắẰằẲẳẴẵẶặẸẹẺẻẼẽẾếỀềỂểỄễỆệỈỉỊịỌọỎỏỐốỒồỔổỖỗỘộỚớỜờỞởỠỡỢợỤụỦủỨứỪừỬửỮữỰựỲỳỴỵỶỷỸỹ'
    s0 = u'AAAAEEEIIOOOOUUYaaaaeeeiioooouuyAaDdIiUuOoUuAaAaAaAaAaAaAaAaAaAaAaAaEeEeEeEeEeEeEeEeIiIiOoOoOoOoOoOoOoOoOoOoOoOoUuUuUuUuUuUuUuUuYyYyYyYy'
    s = ''
    for c in text:
        if c in s1:
            s += s0[s1.index(c)]
        else:
            s += c
    return s

provinces = [
    "An Giang", "Ba Ria Vung Tau", "Bac Lieu", "Bac Kan", "Bac Giang", "Bac Ninh", 
    "Ben Tre", "Binh Duong", "Binh Dinh", "Binh Phuoc", "Binh Thuan", "Ca Mau", 
    "Cao Bang", "Can Tho", "Da Nang", "Dak Lak", "Dak Nong", "Dien Bien", "Dong Nai", 
    "Dong Thap", "Gia Lai", "Ha Giang", "Ha Nam", "Ha Noi", "Ha Tinh", "Hai Duong", 
    "Hai Phong", "Hoa Binh", "Ho Chi Minh", "Hau Giang", "Hung Yen", "Khanh Hoa", 
    "Kien Giang", "Kon Tum", "Lai Chau", "Lao Cai", "Lang Son", "Lam Dong", "Long An", 
    "Nam Dinh", "Nghe An", "Ninh Binh", "Ninh Thuan", "Phu Tho", "Phu Yen", "Quang Binh", 
    "Quang Nam", "Quang Ngai", "Quang Ninh", "Quang Tri", "Soc Trang", "Son La", 
    "Tay Ninh", "Thai Binh", "Thai Nguyen", "Thanh Hoa", "Thua Thien Hue", "Tien Giang", 
    "Tra Vinh", "Tuyen Quang", "Vinh Long", "Vinh Phuc", "Yen Bai"
]

special_mappings = {
    "hcm": "Ho Chi Minh",
    "hcmc": "Ho Chi Minh",
    "sg": "Ho Chi Minh",
    "sai gon": "Ho Chi Minh",
    "hn": "Ha Noi",
    "dn": "Da Nang"
}

# Main mapping function
def clean_and_map_location(loc):
    if pd.isna(loc): return loc
    
    loc_str = str(loc).strip()
    
    if loc_str.lower() in ["remote", "oversea"]:
        return loc_str
        
    loc_clean = remove_diacritics(loc_str).lower()
    
    # Mask street names that share words with province names to avoid false matches
    loc_clean = loc_clean.replace("dien bien phu", "dbp_street")         # Street: Dien Bien Phu
    loc_clean = loc_clean.replace("xa lo ha noi", "xlhn_street")          # Street: Xa Lo Ha Noi
    loc_clean = loc_clean.replace("phuong phu tho", "pt_ward")            # Ward: Phu Tho
    loc_clean = loc_clean.replace("phu tho, ho chi minh", "ho chi minh") # Remove Phu Tho prefix when followed by Ho Chi Minh
    
    found_provinces = set()
    
    # Scan for abbreviations
    for key, val in special_mappings.items():
        if re.search(r'\b' + key + r'\b', loc_clean):
            found_provinces.add(val)
            
    # Scan for all 63 province names
    for p in provinces:
        if p.lower() in loc_clean:
            found_provinces.add(p)
            
    # Join all matched provinces with a comma in alphabetical order
    if len(found_provinces) > 0:
        # sorted() ensures a consistent ordering, e.g. "Ha Noi, Ho Chi Minh" always comes before "Ho Chi Minh, Ha Noi"
        return ", ".join(sorted(list(found_provinces)))
        
    return loc_str 

df['Location'] = df['Location'].apply(clean_and_map_location)

print(f"Location column normalized.")
print(f"Unique locations: {df['Location'].unique().tolist()}")


Location column normalized.
Unique locations: ['Ho Chi Minh', 'Ha Noi', 'Da Nang', 'Bac Ninh', 'Ha Noi, Ho Chi Minh', 'Gia Lai', 'Da Nang, Ho Chi Minh', 'Remote', 'Oversea']


In [16]:
# Impute minimum years of experience. Fill with the median for the same Job Domain
df['Min years of exp'] = df['Min years of exp'].fillna(
    df.groupby('Job Domain')['Min years of exp'].transform('median')
)
# Fallback to the overall market median (used when an entire domain has no experience data at all)
df['Min years of exp'] = df['Min years of exp'].fillna(df['Min years of exp'].median())

In [17]:
def parse_salary(salary_raw):
    s = str(salary_raw).lower().strip()
    # Return NaN for entries that indicate no numeric salary was disclosed
    if any(k in s for k in ['thỏa thuận', 'negotiable', "you'll love it", 
                             'attractive', 'thương lượng', 'competitive']):
        return pd.Series([np.nan, np.nan])
    
    s_clean = s.replace(',', '').replace('.', '')
    numbers = re.findall(r'\d+', s_clean)
    if not numbers: return pd.Series([np.nan, np.nan])
    numbers = [float(n) for n in numbers]
    
    if len(numbers) == 1:
        if any(k in s for k in ['up to', 'lên đến', 'tới', 'max']): 
            return pd.Series([np.nan, numbers[0]])   # Only an upper bound is given
        elif any(k in s for k in ['từ', 'from', 'min']): 
            return pd.Series([numbers[0], np.nan])   # Only a lower bound is given
        else: 
            return pd.Series([numbers[0], numbers[0]])  # Single value: treat as both min and max
    return pd.Series([min(numbers), max(numbers)])


def standardize_currency_full(row, col_name):
    val = row[col_name]
    raw = str(row['Salary_Raw']).lower()
    if pd.isna(val): return np.nan

    has_usd = 'usd' in raw or '$' in raw

    if has_usd and val > 1_000_000:
        # Most likely a mislabeled VND value; keep it as-is
        return val

    if has_usd:
        if val > 20_000:  # A USD value above 20,000 is almost certainly an annual figure
            val = (val / 12) * 25_000  # Convert to monthly, then to VND
        else:
            val = val * 25_000         # Monthly USD: convert directly to VND

    # Recognize 'mil' as a millions keyword alongside 'triệu', 'tr', 'million', etc.
    elif val < 1000 and any(k in raw for k in ['triệu', 'tr', 'm/month', 'mil', 'million']):
        val = val * 1_000_000          # Scale up: value was expressed in millions
    elif 'year' in raw or 'năm' in raw:
        val = val / 12                 # Annual VND: convert to monthly

    return val

df[['Salary_Min_Raw', 'Salary_Max_Raw']] = df['Salary_Raw'].apply(parse_salary)
df['Salary_Min_VND'] = df.apply(lambda r: standardize_currency_full(r, 'Salary_Min_Raw'), axis=1)
df['Salary_Max_VND'] = df.apply(lambda r: standardize_currency_full(r, 'Salary_Max_Raw'), axis=1)

df.drop(columns=['Salary_Min_Raw', 'Salary_Max_Raw'], inplace=True)

print("Missing salary values after Regex parsing:")
print(f"- Salary Min missing: {df['Salary_Min_VND'].isna().sum()} rows")
print(f"- Salary Max missing: {df['Salary_Max_VND'].isna().sum()} rows")

Missing salary values after Regex parsing:
- Salary Min missing: 188 rows
- Salary Max missing: 168 rows


In [18]:
import pandas as pd
import numpy as np

def impute_salary_by_nearest_exp(df, salary_cols):
    df = df.copy()
    
    # STEP 1: BUILD TWO INDEPENDENT REFERENCE POOLS FOR MIN AND MAX
    # POOL FOR SALARY MIN
    pool_min = df.dropna(subset=['Salary_Min_VND']).copy()
    # Remove rows where Min > Max (logical error)
    pool_min = pool_min[~(pool_min['Salary_Min_VND'] > pool_min['Salary_Max_VND'])]
    
    if not pool_min.empty:
        counts_min = pool_min.groupby('Job Domain')['Job Domain'].transform('count')
        limit_min = pool_min.groupby('Job Domain')['Salary_Min_VND'].transform(lambda x: x.quantile(0.80))
        # Apply outlier filter only on the Min column
        pool_min = pool_min[(counts_min < 3) | (pool_min['Salary_Min_VND'] <= limit_min)]

    # POOL FOR SALARY MAX
    pool_max = df.dropna(subset=['Salary_Max_VND']).copy()
    # Remove rows where Min > Max (NaN in Min column is treated as valid and kept)
    pool_max = pool_max[~(pool_max['Salary_Min_VND'] > pool_max['Salary_Max_VND'])]
    
    if not pool_max.empty:
        counts_max = pool_max.groupby('Job Domain')['Job Domain'].transform('count')
        limit_max = pool_max.groupby('Job Domain')['Salary_Max_VND'].transform(lambda x: x.quantile(0.80))
        # Apply outlier filter only on the Max column
        pool_max = pool_max[(counts_max < 3) | (pool_max['Salary_Max_VND'] <= limit_max)]

    # STEP 2: IMPUTE EACH SALARY COLUMN INDEPENDENTLY
    missing_mask = df[salary_cols].isna().any(axis=1)
    missing_rows = df[missing_mask]

    for idx, row in missing_rows.iterrows():
        current_exp = row['Min years of exp']
        job_domain = row['Job Domain']
        
        if pd.isna(current_exp):
            continue

        # 2.1: FIND REFERENCE AND IMPUTE SALARY MIN
        if pd.isna(row['Salary_Min_VND']):
            d_pool_min = pool_min[(pool_min['Job Domain'] == job_domain) & (pool_min.index != idx)].copy()
            if not d_pool_min.empty:
                d_pool_min['_exp_dist'] = (d_pool_min['Min years of exp'] - current_exp).abs()
                
                # Priority 1: All rows within a ±1 year experience window
                nearest_min = d_pool_min[d_pool_min['_exp_dist'] <= 1]
                # Priority 2: Fall back to the single nearest point if no rows found within ±1
                if nearest_min.empty:
                    nearest_min = d_pool_min[d_pool_min['_exp_dist'] == d_pool_min['_exp_dist'].min()]
                
                df.at[idx, 'Salary_Min_VND'] = nearest_min['Salary_Min_VND'].median()

        # 2.2: FIND REFERENCE AND IMPUTE SALARY MAX
        if pd.isna(row['Salary_Max_VND']):
            d_pool_max = pool_max[(pool_max['Job Domain'] == job_domain) & (pool_max.index != idx)].copy()
            if not d_pool_max.empty:
                d_pool_max['_exp_dist'] = (d_pool_max['Min years of exp'] - current_exp).abs()
                
                # Priority 1: All rows within a ±1 year experience window
                nearest_max = d_pool_max[d_pool_max['_exp_dist'] <= 1]
                # Priority 2: Fall back to the single nearest point if no rows found within ±1
                if nearest_max.empty:
                    nearest_max = d_pool_max[d_pool_max['_exp_dist'] == d_pool_max['_exp_dist'].min()]
                
                df.at[idx, 'Salary_Max_VND'] = nearest_max['Salary_Max_VND'].median()

    return df



salary_cols = ['Salary_Min_VND', 'Salary_Max_VND']

df = impute_salary_by_nearest_exp(df, salary_cols)

# Fallback: fill any remaining nulls with the overall market median
for col in salary_cols:
    df[col] = df[col].fillna(df[col].median())

# Round output values
df['Min years of exp'] = np.round(df['Min years of exp'], 1)
df['Salary_Min_VND'] = np.round(df['Salary_Min_VND'], 1)
df['Salary_Max_VND'] = np.round(df['Salary_Max_VND'], 1)

print("\nMissing value counts after imputation:")
print(df[['Min years of exp', 'Salary_Min_VND', 'Salary_Max_VND']].isna().sum())

# Compute the average salary for each row
df['_Temp_Avg_Salary'] = (df['Salary_Min_VND'] + df['Salary_Max_VND']) / 2

salary_stats = df.groupby('Job Domain').agg(
    SALARY_MIN=('Salary_Min_VND', 'min'),
    SALARY_MAX=('Salary_Max_VND', 'max'),
    SALARY_MEAN=('_Temp_Avg_Salary', 'mean')
).round(1)

df = df.drop(columns=['_Temp_Avg_Salary'])

print("\nSALARY SUMMARY STATISTICS BY JOB DOMAIN:")
display(salary_stats)


Missing value counts after imputation:
Min years of exp    0
Salary_Min_VND      0
Salary_Max_VND      0
dtype: int64

SALARY SUMMARY STATISTICS BY JOB DOMAIN:


,SALARY_MIN,SALARY_MAX,SALARY_MEAN
Job Domain,,,
Data & AI,12500000.0,75000000.0,33467741.9
Design & UX,13000000.0,114583333.3,28572222.2
Infrastructure,10000000.0,75000000.0,34093085.1
Management & Analysis,11000000.0,87500000.0,31821052.6
Others,20000000.0,24000000.0,22000000.0
Software Development,15000000.0,125000000.0,35614224.1
Testing & Quality,9000000.0,50000000.0,25129310.3


In [19]:
if 'Salary_Raw' in df.columns:
    df.drop(columns=['Salary_Raw'], inplace=True)

output_dir = "../data/04_processed"
os.makedirs(output_dir, exist_ok=True)

FINAL_OUTPUT_PATH = os.path.join(output_dir, "it_recruitment_processed.csv")
df.to_csv(FINAL_OUTPUT_PATH, index=False)
print(f"- File saved at: {FINAL_OUTPUT_PATH}")
print(f"- Size: {df.shape[0]} rows, {df.shape[1]} columns.")

- File saved at: ../data/04_processed\it_recruitment_processed.csv
- Size: 250 rows, 11 columns.
